<a href="https://colab.research.google.com/github/Kristina4Provorova/SAP_MT_7/blob/main/%D0%97%D0%B0%D0%BD%D1%8F%D1%82%D0%B8%D0%B5_7_(20_11_2026).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Занятие 7. Инструмент «Pipeline API» для управление генерацией перевода и публикация итогового проекта на платформе «GitHub»**

# **Загрузка инструмента «Pipeline API» и моделей OPUS-MT и M2M-100**

Репозиторий инструмента "Pipeline API": https://huggingface.co/docs/transformers.js/pipelines

Репозиторий модели OPUS-MT: https://huggingface.co/Helsinki-NLP/opus-mt-ru-en

Репозиторий модели M2M-100: https://huggingface.co/docs/transformers/model_doc/m2m_100

In [1]:
# --- ШАГ 1: Загрузка библиотеки---
!pip install "transformers<5.0.0" sentencepiece -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [3]:
# --- ШАГ 2: Импорт инструмента "Pipeline API"---
from transformers import pipeline

In [4]:
# --- ШАГ 3: Создание объектов (для двух моделей) ---
# OPUS-MT: одна пара (ru-en)
opus = pipeline("translation_ru_to_en", model="Helsinki-NLP/opus-mt-ru-en")
# M2M: мультиязычная
m2m = pipeline("translation", model="facebook/m2m100_418M", src_lang="ru", tgt_lang="en")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/307M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/307M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

... (8 outputs left out)

vocab.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [6]:
# --- ШАГ 4: Чтение файла ---
with open("source_text.txt", "r", encoding="utf-8") as f:# ЗДЕСЬ НЕОБХОДИМО БУДЕТ ЗАМЕНЯТЬ ФАЙЛ ДЛЯ 1 ЗАДАНИЯ !!!!!!!!!!!!!
    text = f.read()

In [8]:
# --- ШАГ 5: Перевод для модели OPUS-MT---
# OPUS-MT: Greedy
opus_greedy = opus(text)[0]["translation_text"]
# OPUS-MT: Beam Search
opus_beam = opus(text, num_beams=5)[0]["translation_text"]
# OPUS-MT: Семплирование с температурой 0.7
opus_sample_1_2 = opus(text, do_sample=False, temperature=0.1)[0]["translation_text"]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:
# --- ШАГ 5.1.: Перевод для модели M2M-100---
# M2M: нужно указать язык оригинала
m2m.tokenizer.src_lang = "ru"
# M2M: Greedy
m2m_greedy = m2m(text)[0]["translation_text"]
# M2M: Beam Search
m2m_beam = m2m(text, num_beams=5)[0]["translation_text"]
# M2M: Семплирование с температурой 0.7
m2m_sample_1_2 = m2m(text, do_sample=False, temperature=0.1)[0]["translation_text"]

In [ ]:
# --- ШАГ 6: Сохранение результатов (переводы модели OPUS-MT)---
with open("opus_greedy.txt", "w", encoding="utf-8") as f: f.write(opus_greedy)

In [ ]:
with open("opus_beam.txt", "w", encoding="utf-8") as f: f.write(opus_beam)

In [ ]:
with open("opus_sample_1.2.txt", "w", encoding="utf-8") as f: f.write(opus_sample_1_2)

In [ ]:
# --- ШАГ 6.1.: Сохранение результатов (переводы модели M2M-100)---
with open("m2m_greedy.txt", "w", encoding="utf-8") as f: f.write(m2m_greedy)

In [ ]:
with open("m2m_beam.txt", "w", encoding="utf-8") as f: f.write(m2m_beam)

In [ ]:
with open("m2m_sample.txt", "w", encoding="utf-8") as f: f.write(m2m_sample)

In [ ]:
with open("m2m_sample_1.2.txt", "w", encoding="utf-8") as f: f.write(m2m_sample_1_2)

In [ ]:
# --- ШАГ 7: Вывод результатов (убеждаемся, что все переводы выполнены и сохранены)---
print("Переводы сохранены")

Варианты: pipeline, pipeline, task, num_beams, "translation_text".

# **Задание 1. Оценка качества переводов моделями.**

Необходимо в командах перевести три текста разных жанров с помощью двух моделей (OPUS-MT и M2M-100) с разными параметрами стратегий генерации перевода (представлены в таблице ниже) и оценить качество полученных переводов с помощью BLEU и BERTScore.

**Наборы параметров для сравнения**

| Набор | Параметры |
| :---: | :--- |
| **A** | `num_beams=1` (Greedy) |
| **B** | `num_beams=5` |
| **C** | `do_sample=True`, `temperature=0.7` |
| **D** | `do_sample=True`, `temperature=1.2` |

---


Ссылка на таблицу для заполнения: https://docs.google.com/spreadsheets/d/1bhKtwxIkL9domK7goA2sUxnRuqVPIIzv9EBL6Dq8YHk/edit?usp=sharing


---

**❗** Не забудьте поменять названия файлов с переводами, чтобы они не дублировались. Также помните, что при переводе необязательно одновременно выводить результаты всех ячеек с файлами переводов!

**❗** Не забудьте при оценке качества каждого текста указать соответствующий эталонный перевод (например, для "news_source.txt" эталонный перевод будет храниться в файле "news_reference.txt")




# **Оценка качества переводов моделей OPUS-MT и M2M-100 с помощью метрик BLEU и BERTScore**

In [ ]:
# --- 1. Устанавливаем библиотеки ---
!pip install sacrebleu
!pip install bert-score
!pip install pandas

In [ ]:
# --- 2. Импортируем инструменты для подсчета метрик ---
from sacrebleu.metrics import BLEU
from bert_score import score
import pandas as pd

In [ ]:
# --- 3. Читаем эталонный перевод из файла (ДЛЯ КАЖДОГО ЖАНРА БУДЕТ СВОЙ ЭТАЛОННЫЙ ПЕРЕВОД!!!!!) ---
with open("news_reference.txt", "r", encoding="utf-8") as f:# НЕ ЗАБУДЬТЕ ЗДЕСЬ ЗАМЕНИТЬ НАЗВАНИЕ ФАЙЛА ЭТАЛОННОГО ПЕРЕВОДА!!!!!!!!!!!!!!!!
    reference = f.read()

In [ ]:
# --- 4. Читаем переводы моделей ---

# Чтение переводов OPUS-MT
# Файл: opus_greedy.txt (Greedy, num_beams=1)
with open("opus_greedy.txt", "r", encoding="utf-8") as f:
    opus_greedy = f.read()

# Файл: opus_beam.txt (Beam Search, num_beams=5)
with open("opus_beam.txt", "r", encoding="utf-8") as f:
    opus_beam = f.read()

# Файл: opus_sample.txt (семплирование, temperature=0.7)
with open("opus_sample.txt", "r", encoding="utf-8") as f:
    opus_sample = f.read()

# Файл: opus_sample_1.2.txt (семплирование, temperature=1.2)
with open("opus_sample_1.2.txt", "r", encoding="utf-8") as f:
    opus_sample_1_2 = f.read()

# Чтение переводов M2M-100
# Файл: m2m_greedy.txt (Greedy, num_beams=1)
with open("m2m_greedy.txt", "r", encoding="utf-8") as f:
    m2m_greedy = f.read()

# Файл: m2m_beam.txt (Beam Search, num_beams=5)
with open("m2m_beam.txt", "r", encoding="utf-8") as f:
    m2m_beam = f.read()

# Файл: m2m_sample.txt (семплирование, temperature=0.7)
with open("m2m_sample.txt", "r", encoding="utf-8") as f:
    m2m_sample = f.read()

# Файл: m2m_sample_1.2.txt (семплирование, temperature=1.2)
with open("m2m_sample_1.2.txt", "r", encoding="utf-8") as f:
    m2m_sample_1_2 = f.read()

In [ ]:
# --- 5. Считаем BLEU для каждой модели ---
bleu = BLEU()

# Считаем BLEU для OPUS-MT
bleu_opus_greedy = bleu.sentence_score(opus_greedy, [reference]).score
bleu_opus_beam = bleu.sentence_score(opus_beam, [reference]).score
bleu_opus_sample = bleu.sentence_score(opus_sample, [reference]).score
bleu_opus_sample_1_2 = bleu.sentence_score(opus_sample_1_2, [reference]).score

# Считаем BLEU для M2M-100
bleu_m2m_greedy = bleu.sentence_score(m2m_greedy, [reference]).score
bleu_m2m_beam = bleu.sentence_score(m2m_beam, [reference]).score
bleu_m2m_sample = bleu.sentence_score(m2m_sample, [reference]).score
bleu_m2m_sample_1_2 = bleu.sentence_score(m2m_sample_1_2, [reference]).score

In [ ]:
# --- 6. Считаем BERTScore для двух моделей ---
candidates = [
    opus_greedy,
    opus_beam,
    opus_sample,
    opus_sample_1_2,
    m2m_greedy,
    m2m_beam,
    m2m_sample,
    m2m_sample_1_2
]

references = [reference] * 8  # тот же эталон 8 раз (по числу кандидатов)

P, R, F1 = score(candidates, references, lang="en", batch_size=8)

In [ ]:
# --- 7. Создаём таблицу с оценкой BLEU и BERTScore ---
df = pd.DataFrame({
    "Система": [
        "OPUS Greedy",
        "OPUS Beam",
        "OPUS Sample (T=0.7)",
        "OPUS Sample (T=1.2)",
        "M2M Greedy",
        "M2M Beam",
        "M2M Sample (T=0.7)",
        "M2M Sample (T=1.2)"
    ],
    "BLEU": [
        bleu_opus_greedy,
        bleu_opus_beam,
        bleu_opus_sample,
        bleu_opus_sample_1_2,
        bleu_m2m_greedy,
        bleu_m2m_beam,
        bleu_m2m_sample,
        bleu_m2m_sample_1_2
    ],
    "BERTScore F1": [
        F1[0].item(),
        F1[1].item(),
        F1[2].item(),
        F1[3].item(),
        F1[4].item(),
        F1[5].item(),
        F1[6].item(),
        F1[7].item()
    ]
})

In [ ]:
# --- 8. Сохраняем таблицу с результатами ---
print(df)
df.to_csv("metrics_results.csv", index=False, encoding="utf-8-sig")

# **Задание 2. Подготовка файла README для репозитория**
Необходимо заполнить шаблон описания проекта.

In [ ]:
# Название проекта
Пайплайн для работы с нейросетевым автоматическим переводом и для оценки качества переводов.

## Автор
[Ваше Имя и Фамилия или ник пользователя]

## О проекте
Данный проект представляет собой полный пайплайн для автоматического перевода текстов с русского на английский язык и последующей оценки качества полученных переводов.

## Используемые инструменты и модели
- **Среда разработки:**
- **API-Библиотеки NMT:**
- **Модели NMT:**
- **Стратегии генерации:**
- **Оценка качества:**

## Структура проекта

## Как запустить проект

## Результаты
- Лучшая библиотека и модель для новостного текста: [название]
- Лучшая библиотека и модель для юридического текста: [название]
- Лучшая библиотека и модель для художественного текста: [название]

In [ ]:
from google.colab import userdata

# 1.Достаем токен из секретов
token = userdata.get('НАЗВАНИЕ ВАШЕГО ТОКЕНА ИЛИ API-КЛЮЧА')

# 2. Указываем ТОЛЬКО путь к репозиторию (без https:// и без решетки #)
repo_path = "github.com/Kristina4Provorova/sap"

# 3. Клонируем
!git clone https://{token}@{repo_path}

# **Загрузка репозитория обратно с платформы «GitHub» в блокнот «Google Colab»**

In [ ]:
from google.colab import userdata

# 1.Достаем токен из секретов
token = userdata.get('НАЗВАНИЕ ВАШЕГО ТОКЕНА ИЛИ API-КЛЮЧА')

# 2. Указываем ТОЛЬКО путь к репозиторию (без https:// и без решетки #)
repo_path = "ССЫЛКА НА ВАШ РЕПОЗИТОРИЙ"

# 3. Клонируем
!git clone https://{token}@{repo_path}